# Heart Disease Prediction — Data Mining
**University of Jeddah | Data Science Department | 2025**

Complete data mining pipeline on 2,182 patient records from 5 combined public datasets.

**Pipeline:**
```
EDA → Data Cleaning → Decision Tree (12 configs) → K-Means Clustering → DT per Cluster
```

## 1. Setup & Import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print('Libraries loaded successfully.')

## 2. Load Data

In [ ]:
df = pd.read_csv('merged_heart_dataset.csv')

print(f'Shape  : {df.shape}')
print(f'\nTarget distribution:')
print(df['target'].value_counts())
print(f'\nMissing values: {df.isnull().sum().sum()}')
df.head()

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Age distribution
axes[0].hist(df['age'], bins=12, color='steelblue', edgecolor='white')
axes[0].set_title('Age Distribution')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Frequency')

# Target distribution
df['target'].value_counts().plot(kind='bar', ax=axes[1],
    color=['steelblue', 'tomato'])
axes[1].set_title('Heart Disease Distribution')
axes[1].set_xticklabels(['Low Risk (0)', 'High Risk (1)'], rotation=0)

# Correlation heatmap
corr = df.corr()
sns.heatmap(corr[['target']].sort_values('target', ascending=False),
            annot=True, fmt='.2f', cmap='RdYlGn', ax=axes[2],
            cbar=False)
axes[2].set_title('Feature Correlation with Target')

plt.tight_layout()
plt.show()

In [ ]:
# Boxplots for key numerical features
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
num_cols = ['trestbps', 'chol', 'thalachh']
labels   = ['Resting Blood Pressure', 'Cholesterol', 'Max Heart Rate']

for i, (col, label) in enumerate(zip(num_cols, labels)):
    axes[i].boxplot(df[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.6))
    axes[i].set_title(label)
    axes[i].set_ylabel('Value')

plt.tight_layout()
plt.show()

print('Summary statistics:')
print(df[num_cols].describe().round(2))

## 4. Data Cleaning

In [ ]:
df_clean = df.copy()

# Remove outliers using IQR for numerical columns
num_cols = ['trestbps', 'chol', 'thalachh', 'oldpeak']
for col in num_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    df_clean = df_clean[
        (df_clean[col] >= Q1 - 1.5 * IQR) &
        (df_clean[col] <= Q3 + 1.5 * IQR)
    ]

df_clean = df_clean.reset_index(drop=True)
print(f'Before cleaning : {df.shape[0]} records')
print(f'After cleaning  : {df_clean.shape[0]} records')
print(f'Removed         : {df.shape[0] - df_clean.shape[0]} records')

## 5. Decision Tree Classification

12 configurations compared across 3 splitting criteria × 4 leaf sizes.

In [ ]:
X = df_clean.drop('target', axis=1)
y = df_clean['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

# ── Compare 12 configurations ──────────────────────────────────
criteria   = ['gini', 'entropy']  # entropy = Information Gain
leaf_sizes = [0.035, 0.04, 0.05, 0.06]
results    = []

for criterion in criteria:
    for leaf in leaf_sizes:
        min_leaf = max(1, int(leaf * len(X_train)))
        dt = DecisionTreeClassifier(
            criterion=criterion,
            min_samples_leaf=min_leaf,
            random_state=42
        )
        dt.fit(X_train, y_train)
        acc  = accuracy_score(y_test, dt.predict(X_test))
        cv   = cross_val_score(dt, X, y, cv=5, scoring='accuracy').mean()
        name = f"{'Info Gain' if criterion=='entropy' else 'Gini'} · {int(leaf*100)}% leaf"
        results.append({'Config': name, 'Test Accuracy': acc, 'CV Accuracy': cv,
                        'Leaves': dt.get_n_leaves(), 'criterion': criterion, 'leaf': leaf})

results_df = pd.DataFrame(results).sort_values('Test Accuracy', ascending=False)
print('Decision Tree Results:')
print(results_df[['Config','Test Accuracy','CV Accuracy','Leaves']].to_string(index=False))

In [ ]:
# ── Accuracy comparison bar chart ──────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['gold' if i == 0 else 'steelblue' for i in range(len(results_df))]
bars = ax.barh(results_df['Config'], results_df['Test Accuracy'], color=colors)
ax.set_xlabel('Test Accuracy')
ax.set_title('Decision Tree — All Configurations Compared')
ax.set_xlim(0.75, 0.90)
for bar, val in zip(bars, results_df['Test Accuracy']):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Best model evaluation ──────────────────────────────────────
best = results_df.iloc[0]
min_leaf_best = max(1, int(best['leaf'] * len(X_train)))

best_dt = DecisionTreeClassifier(
    criterion=best['criterion'],
    min_samples_leaf=min_leaf_best,
    random_state=42
)
best_dt.fit(X_train, y_train)
y_pred = best_dt.predict(X_test)

print('=' * 50)
print(f'BEST MODEL: {best["Config"]}')
print('=' * 50)
print(classification_report(y_test, y_pred, digits=4))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low Risk', 'High Risk'],
            yticklabels=['Low Risk', 'High Risk'])
plt.title(f'Confusion Matrix — {best["Config"]}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature Importance ─────────────────────────────────────────
feat_imp = pd.DataFrame({
    'Feature'   : X.columns,
    'Importance': best_dt.feature_importances_
}).sort_values('Importance', ascending=True)

feat_imp.plot(kind='barh', x='Feature', y='Importance',
              figsize=(10, 6), color='steelblue', legend=False)
plt.title('Feature Importance — Best Decision Tree')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 6. K-Means Patient Segmentation

Clustering patients into distinct groups to discover hidden subpopulations.

In [ ]:
# ── Elbow Method to select K ───────────────────────────────────
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

inertias = []
K_range  = range(2, 8)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_range, inertias, 'bo-', markersize=8)
plt.axvline(x=3, color='red', linestyle='--', alpha=0.7, label='Selected K=3')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method — Optimal K Selection')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── K=3 Clustering ────────────────────────────────────────────
km3 = KMeans(n_clusters=3, random_state=42, n_init=10)
df_clean['cluster'] = km3.fit_predict(X_scaled)

print('Cluster Distribution:')
print(df_clean['cluster'].value_counts().sort_index())

print('\nCluster Profiles (mean values):')
print(df_clean.groupby('cluster')[['age','trestbps','chol','thalachh','target']]
      .mean().round(2))

In [ ]:
# ── Cluster visualization ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {0: 'steelblue', 1: 'tomato', 2: 'seagreen'}

# Age vs Max Heart Rate
for cluster in [0, 1, 2]:
    mask = df_clean['cluster'] == cluster
    axes[0].scatter(df_clean[mask]['age'], df_clean[mask]['thalachh'],
                    label=f'Cluster {cluster}', alpha=0.5, s=20,
                    color=colors[cluster])
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Max Heart Rate')
axes[0].set_title('K=3 Clusters: Age vs Max Heart Rate')
axes[0].legend()

# Heart disease rate per cluster
churn_rate = df_clean.groupby('cluster')['target'].mean()
churn_rate.plot(kind='bar', ax=axes[1],
                color=[colors[i] for i in churn_rate.index])
axes[1].set_title('Heart Disease Rate per Cluster')
axes[1].set_ylabel('Rate')
axes[1].set_xticklabels([f'Cluster {i}' for i in churn_rate.index], rotation=0)

plt.tight_layout()
plt.show()

## 7. Decision Tree per Cluster

In [ ]:
# ── Train Decision Tree on each cluster ───────────────────────
print('Decision Tree Results per Cluster:')
print('=' * 50)

for cluster in sorted(df_clean['cluster'].unique()):
    cluster_data = df_clean[df_clean['cluster'] == cluster]
    Xc = cluster_data.drop(['target', 'cluster'], axis=1)
    yc = cluster_data['target']

    if len(yc.unique()) < 2 or len(Xc) < 20:
        print(f'Cluster {cluster}: insufficient data')
        continue

    Xc_train, Xc_test, yc_train, yc_test = train_test_split(
        Xc, yc, test_size=0.2, stratify=yc, random_state=42)

    dt_c = DecisionTreeClassifier(
        criterion='entropy',
        min_samples_leaf=max(1, int(0.035 * len(Xc_train))),
        random_state=42
    )
    dt_c.fit(Xc_train, yc_train)
    acc = accuracy_score(yc_test, dt_c.predict(Xc_test))

    top_features = pd.Series(
        dt_c.feature_importances_, index=Xc.columns
    ).sort_values(ascending=False).head(3)

    print(f'\nCluster {cluster} ({len(cluster_data)} patients)')
    print(f'  Accuracy    : {acc:.3f}')
    print(f'  Top features: {list(top_features.index)}')

## 8. Results Summary

### Decision Tree

| Metric | Score |
|--------|-------|
| Best Accuracy | 85% |
| Overall Score | 0.87 |
| Lift (3rd decile) | 1.20 |
| Best Config | Information Gain + 3.5% min leaf |

### K-Means Clustering (K=3)

| Cluster | Profile | Key Driver |
|---------|---------|------------|
| Cluster 0 | Moderate risk · Middle-aged | Thalassemia type |
| Cluster 1 | Higher risk · Older | Chest pain type |
| Cluster 2 | High chest pain risk | ST slope |

**Key Findings:**
- Information Gain + 3.5% leaf consistently outperformed all other configurations
- K=3 clustering revealed 3 clinically distinct patient subgroups
- Thalassemia type (thal) and chest pain type (cp) are the strongest predictors
- Decision trees per cluster provided interpretable clinical insights